# TypiClust Modified: Cosine Similarity Typicality

## Modification

This notebook replaces the **Euclidean distance-based** typicality measure from
Hacohen et al. (ICML 2022) with a **cosine similarity-based** variant:

| | Original (paper) | Modified (this notebook) |
|---|---|---|
| **Formula** | Typicality(x) = (1/K * sum \|\|x - x_i\|\|_2)^{-1} | Typicality(x) = 1/K * sum cos_sim(x, x_i) |
| **KNN metric** | Euclidean distance | Cosine distance (= 1 - cos_sim) |
| **Score range** | (0, inf) | (-1, 1] |
| **Aggregation** | Inverse of mean distance | Mean of similarities |

### Rationale

SimCLR features are L2-normalised and live on a **unit hypersphere**.  While
Euclidean distance between unit vectors is a monotone transform of cosine
distance (||a-b||^2 = 2 - 2*cos(a,b)), the *aggregation* differs:

- **Original**: takes the mean of distances, then inverts (1/x).  The 1/x
  non-linearity amplifies the influence of a single large distance, making
  the score sensitive to outlier neighbours.
- **Modified**: takes the mean of similarities directly.  This is a linear
  aggregation, so outlier neighbours have proportional (not amplified)
  influence, potentially making cosine typicality **more robust**.

### What stays the same

- SimCLR features (same checkpoint, no retraining)
- K-Means clustering step (identical clusters each round)
- Uncovered-cluster selection logic
- Classifier (ResNet-18 from scratch, same hyperparameters)
- Experiment protocol: B=10, 5 rounds, 3+ seeds

---

**Prerequisites:** run `tpcrp_original.ipynb` first to generate:
- `results/simclr_checkpoint.pt`
- `results/train_features.npy` / `results/test_features.npy`
- `results/train_labels.npy` / `results/test_labels.npy`

---
## Section 1: Setup & Imports

In [ ]:
%%time
import os, sys, time, json, warnings, random, glob
from pathlib import Path
from collections import defaultdict

# 1. Install dependencies
#!pip install -q torch torchvision scikit-learn numpy matplotlib scipy tqdm

# 2. Setup Repository (Environment Agnostic)
#REPO_URL = "https://github.com/UlvisTurkers/machine-learning-coursework-2.git"

# On Kaggle/Local, we'll just use the current working directory
#REPO_NAME = "machine-learning-coursework-2"
#REPO_DIR = os.path.abspath(REPO_NAME)

if not os.path.exists(REPO_DIR):
    print(f"Cloning {REPO_NAME}...")
    !git clone -b develop {REPO_URL} {REPO_DIR}
else:
    print(f"Updating {REPO_NAME}...")
    !cd {REPO_DIR} && git fetch origin && git checkout develop && git pull origin develop

# 3. Pathing Logic
%cd {REPO_DIR}
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# 4. Standard ML Imports
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats
from sklearn.manifold import TSNE

warnings.filterwarnings('ignore')

# 5. Project-Specific Imports
try:
    from src.utils import set_seed, load_cifar10, extract_features, save_results, load_results, log
    from src.simclr import SimCLRModel, train_simclr, load_simclr_model, get_features
    from src.active_learning import TypiClust, RandomSelection
    from src.classifier import CIFARClassifier
    print('All imports OK.')
except ModuleNotFoundError as e:
    print(f"Error: {e}")
    print("Ensure your 'src' folder exists in the repository on the 'develop' branch.")

In [ ]:
GLOBAL_SEED = 42
set_seed(GLOBAL_SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device :', DEVICE)
if DEVICE.type == 'cuda':
    print('GPU    :', torch.cuda.get_device_name(0))

RESULTS_DIR = Path('../results')
PLOTS_DIR   = Path('../plots')
RESULTS_DIR.mkdir(exist_ok=True)
PLOTS_DIR.mkdir(exist_ok=True)

In [ ]:
from torch.utils.data import DataLoader

# 1. Load the raw datasets
train_dataset, test_dataset = load_cifar10(root='./data')

# 2. Wrap them in PyTorch DataLoaders (this handles the batching)
train_loader = DataLoader(train_dataset, batch_size=256, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False)

# 3. Load the pre-trained weights (the function initializes the model for you!)
simclr_model = load_simclr_model("./results/simclr_checkpoint.pt")

# 4. Move to GPU and set to evaluation mode
simclr_model = simclr_model.to(DEVICE)
simclr_model.eval() 

print("Model and DataLoaders ready for extraction!")

---
## Section 2: Load SAME SimCLR Features (no retraining)

In [ ]:
# ── Results Directory and Feature Loading ──────────────────────────────────
# Ensure the results directory exists locally
RESULTS_DIR = Path('./results')
RESULTS_DIR.mkdir(exist_ok=True)

# Define local file paths for features and labels
f_train_feats  = RESULTS_DIR / 'train_features.npy'
f_train_labels = RESULTS_DIR / 'train_labels.npy'
f_test_feats   = RESULTS_DIR / 'test_features.npy'
f_test_labels  = RESULTS_DIR / 'test_labels.npy'

# Check if pre-extracted features exist; if not, run extraction
# Note: This requires 'simclr_model', 'train_loader', and 'test_loader' to be defined
if not f_train_feats.exists():
    print("Notice: .npy files not found in ./results. Running extraction...")
    
    # Extract features using the model and data loaders defined in previous cells
    train_feats, train_labels = extract_features(simclr_model, train_loader, DEVICE)
    test_feats, test_labels   = extract_features(simclr_model, test_loader, DEVICE)

    # Save to current environment storage to allow quick loading later
    np.save(f_train_feats, train_feats)
    np.save(f_train_labels, train_labels)
    np.save(f_test_feats, test_feats)
    np.save(f_test_labels, test_labels)
    print("Notice: Extraction finished and files saved.")
else:
    print("Notice: Loading features from existing ./results cache...")
    train_feats  = np.load(str(f_train_feats))
    train_labels = np.load(str(f_train_labels))
    test_feats   = np.load(str(f_test_feats))
    test_labels  = np.load(str(f_test_labels))

# ── Dataset Metadata ────────────────────────────────────────────────────────
CIFAR10_CLASSES = [
    'airplane', 'automobile', 'bird', 'cat', 'deer',
    'dog', 'frog', 'horse', 'ship', 'truck',
]
NUM_CLASSES = len(CIFAR10_CLASSES)

# ── Verification and Stats ──────────────────────────────────────────────────
norms = np.linalg.norm(train_feats, axis=1)
print('-' * 40)
print('Train features : {}   norm mean={:.4f}'.format(train_feats.shape, norms.mean()))
print('Test features  : {}'.format(test_feats.shape))
print('Features are L2-normalised: {}'.format('yes' if np.allclose(norms, 1.0, atol=1e-3) else 'NO'))

In [ ]:
train_dataset, test_dataset = load_cifar10(root='../data')
print('Datasets loaded (needed for CIFARClassifier image augmentation).')

---
## Section 3: Experiment Configuration

**Identical protocol** to `tpcrp_original.ipynb` — the only difference is
the typicality scoring function inside the modified selector.

In [ ]:
BUDGET_B     = 10
N_ROUNDS     = 5
N_SEEDS      = 3       # same as original; increase to 10 for paper quality
CLF_EPOCHS   = 200
CLF_BATCH    = 64
MAX_CLUSTERS = 500
SEEDS        = list(range(N_SEEDS))

BUDGETS = [BUDGET_B * (r + 1) for r in range(N_ROUNDS)]
print('Budgets     :', BUDGETS)
print('Seeds       :', SEEDS)
total_runs = len(SEEDS) * N_ROUNDS * 3   # 3 strategies this time
print('Total classifier training runs: {} (3 strategies x {} seeds x {} rounds)'.format(
    total_runs, N_SEEDS, N_ROUNDS))

---
## Section 4: Shared AL Loop

This is the **exact same** experiment driver as `tpcrp_original.ipynb`.

In [ ]:
def run_al_experiment(strategy_name, make_selector_fn, seeds):
    all_results = {}

    for seed in seeds:
        print('\n' + '=' * 60)
        print('  {}  |  seed={}'.format(strategy_name, seed))
        print('=' * 60)
        set_seed(seed)

        selector    = make_selector_fn(seed)
        clf         = CIFARClassifier(device=DEVICE, seed=seed, num_workers=2)
        labeled_idx = np.array([], dtype=np.int64)
        seed_records = []

        for rnd in range(N_ROUNDS):
            new_idx = selector.select(
                BUDGET_B,
                labeled_indices=labeled_idx if len(labeled_idx) > 0 else None,
            )
            labeled_idx = np.concatenate([labeled_idx, new_idx]).astype(np.int64)
            budget_now = len(labeled_idx)

            selected_classes = [CIFAR10_CLASSES[train_labels[i]] for i in new_idx]
            print('\n  Round {}/{} | labeled={}'.format(rnd + 1, N_ROUNDS, budget_now))
            print('  Selected classes:', selected_classes)

            t0 = time.time()
            train_hist = clf.train(
                labeled_idx, train_dataset,
                epochs=CLF_EPOCHS, batch_size=CLF_BATCH,
                seed=seed, verbose=False,
            )
            train_time = time.time() - t0

            eval_res = clf.evaluate(test_dataset)
            test_acc = eval_res['accuracy']

            print('  Test accuracy: {:.2f}%  (train {:.1f}s)'.format(test_acc, train_time))

            seed_records.append({
                'round':            rnd + 1,
                'budget':           int(budget_now),
                'test_accuracy':    float(test_acc),
                'per_class_acc':    eval_res['per_class_acc'].tolist(),
                'selected_indices': new_idx.tolist(),
                'final_train_loss': float(train_hist['train_loss'][-1]),
                'final_train_acc':  float(train_hist['train_acc'][-1]),
            })

        all_results[seed] = seed_records

    return all_results

In [ ]:
def aggregate(results_dict):
    seeds_sorted = sorted(results_dict.keys())
    acc_matrix = np.array([
        [r['test_accuracy'] for r in results_dict[s]]
        for s in seeds_sorted
    ])
    budgets = [r['budget'] for r in results_dict[seeds_sorted[0]]]
    n_seeds = len(seeds_sorted)
    return {
        'mean':    acc_matrix.mean(axis=0),
        'std':     acc_matrix.std(axis=0, ddof=1) if n_seeds > 1 else np.zeros(len(budgets)),
        'matrix':  acc_matrix,
        'budgets': budgets,
    }

---
## Section 5: Run All Three Strategies

| Strategy | Typicality metric | KNN metric | What changed? |
|---|---|---|---|
| **TypiClust (original)** | Inverse mean Euclidean distance | Euclidean | Nothing (paper baseline) |
| **TypiClust (cosine)** | Mean cosine similarity | Cosine | Typicality scoring only |
| **Random** | N/A | N/A | Nothing (random baseline) |

In [ ]:
%%time
# 1. Original TypiClust (Euclidean typicality)
print('='*60)
print(' ORIGINAL TypiClust (Euclidean typicality)')
print('='*60)

def make_original(seed):
    return TypiClust(
        features=train_feats, max_clusters=MAX_CLUSTERS,
        min_cluster_size=5, seed=seed,
    )

results_original = run_al_experiment('TypiClust-Euclidean', make_original, SEEDS)
save_results(
    {str(k): v for k, v in results_original.items()},
    str(RESULTS_DIR / 'modified_original_results.json'),
)
print('\nOriginal TypiClust complete.')

In [ ]:
%%time
# 2. Modified TypiClust (Cosine typicality)
# ONLY the typicality scoring changes; clustering and selection logic are identical.
print('='*60)
print(' MODIFIED TypiClust (Cosine similarity typicality)')
print('='*60)

def make_cosine(seed):
    return TypiClustCosine(
        features=train_feats, max_clusters=MAX_CLUSTERS,
        min_cluster_size=5, seed=seed,
    )

results_cosine = run_al_experiment('TypiClust-Cosine', make_cosine, SEEDS)
save_results(
    {str(k): v for k, v in results_cosine.items()},
    str(RESULTS_DIR / 'modified_cosine_results.json'),
)
print('\nModified TypiClust (cosine) complete.')

In [ ]:
%%time
# 3. Random baseline
print('='*60)
print(' RANDOM BASELINE')
print('='*60)

def make_random(seed):
    return RandomSelection(train_feats, seed=seed)

results_random = run_al_experiment('Random', make_random, SEEDS)
save_results(
    {str(k): v for k, v in results_random.items()},
    str(RESULTS_DIR / 'modified_random_results.json'),
)
print('\nRandom baseline complete.')

In [ ]:
# aggregate
agg_orig = aggregate(results_original)
agg_cos  = aggregate(results_cosine)
agg_rnd  = aggregate(results_random)
budgets_arr = np.array(agg_orig['budgets'])
print('Aggregation complete.')

---
## Section 6: Comparison Table

In [ ]:
header = '{:>8s}  {:>20s}  {:>20s}  {:>16s}'.format(
    'Budget', 'Original (Euclid)', 'Modified (Cosine)', 'Random')
print(header)
print('-' * len(header))

for i, b in enumerate(budgets_arr):
    orig_str = '{:.2f} +/- {:.2f}'.format(agg_orig['mean'][i], agg_orig['std'][i])
    cos_str  = '{:.2f} +/- {:.2f}'.format(agg_cos['mean'][i],  agg_cos['std'][i])
    rnd_str  = '{:.2f} +/- {:.2f}'.format(agg_rnd['mean'][i],  agg_rnd['std'][i])
    print('{:>8d}  {:>20s}  {:>20s}  {:>16s}'.format(b, orig_str, cos_str, rnd_str))

print()
print('Delta (cosine - original) at final budget: {:+.2f}%'.format(
    agg_cos['mean'][-1] - agg_orig['mean'][-1]))

In [ ]:
# per-seed breakdown
for seed in SEEDS:
    orig_accs = ['{:.1f}'.format(r['test_accuracy']) for r in results_original[seed]]
    cos_accs  = ['{:.1f}'.format(r['test_accuracy']) for r in results_cosine[seed]]
    rnd_accs  = ['{:.1f}'.format(r['test_accuracy']) for r in results_random[seed]]
    print('Seed {}  Original: {}  Cosine: {}  Random: {}'.format(
        seed, orig_accs, cos_accs, rnd_accs))

---
## Section 7: Plots

In [ ]:
plt.rcParams.update({
    'font.size':       11,
    'axes.titlesize':  13,
    'axes.labelsize':  11,
    'legend.fontsize': 10,
    'figure.dpi':      120,
})

ORIG_COLOR = '#1f77b4'   # blue
COS_COLOR  = '#2ca02c'   # green
RND_COLOR  = '#ff7f0e'   # orange

In [ ]:
# plot 1: Accuracy vs Budget (3 curves with std shading)
fig, ax = plt.subplots(figsize=(8, 5))

for agg, label, color, marker in [
    (agg_orig, 'TypiClust (Euclidean)', ORIG_COLOR, 'o'),
    (agg_cos,  'TypiClust (Cosine)',    COS_COLOR,  's'),
    (agg_rnd,  'Random',                RND_COLOR,  '^'),
]:
    mu  = agg['mean']
    std = agg['std']
    ax.plot(budgets_arr, mu, marker=marker, linewidth=2, color=color, label=label)
    ax.fill_between(budgets_arr, mu - std, mu + std, alpha=0.15, color=color)

ax.set_xlabel('Cumulative labeled budget')
ax.set_ylabel('Test accuracy (%)')
ax.set_title('TypiClust: Euclidean vs. Cosine Typicality -- CIFAR-10')
ax.set_xticks(budgets_arr)
ax.legend(framealpha=0.9)
ax.grid(True, linestyle='--', alpha=0.4)
ax.margins(x=0.05)

plt.tight_layout()
plt.savefig(str(PLOTS_DIR / 'modified_accuracy_vs_budget.pdf'), bbox_inches='tight')
plt.show()
print('Saved -> plots/modified_accuracy_vs_budget.pdf')

In [ ]:
# plot 2: Bar chart at final budget
methods = ['TypiClust\n(Euclidean)', 'TypiClust\n(Cosine)', 'Random']
means   = [agg_orig['mean'][-1], agg_cos['mean'][-1], agg_rnd['mean'][-1]]
stds    = [agg_orig['std'][-1],  agg_cos['std'][-1],  agg_rnd['std'][-1]]
colors  = [ORIG_COLOR, COS_COLOR, RND_COLOR]

fig, ax = plt.subplots(figsize=(5.5, 4.5))
bars = ax.bar(methods, means, color=colors, width=0.5,
              yerr=stds, capsize=6, error_kw=dict(linewidth=1.5))

for bar, m, s in zip(bars, means, stds):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + s + 0.3,
            '{:.2f}%'.format(m),
            ha='center', va='bottom', fontsize=10)

ax.set_ylabel('Test accuracy (%)')
ax.set_title('Final accuracy at budget={}'.format(BUDGETS[-1]))
ax.set_ylim(0, max(means) + max(stds) + 8)
ax.grid(True, axis='y', linestyle='--', alpha=0.4)

plt.tight_layout()
plt.savefig(str(PLOTS_DIR / 'modified_final_bar.pdf'), bbox_inches='tight')
plt.show()
print('Saved -> plots/modified_final_bar.pdf')

In [ ]:
# plot 3: Difference plot (Cosine - Original) per budget
deltas = agg_cos['mean'] - agg_orig['mean']

fig, ax = plt.subplots(figsize=(7, 3.5))
bar_colors = [COS_COLOR if d >= 0 else ORIG_COLOR for d in deltas]
ax.bar(budgets_arr, deltas, color=bar_colors, width=3.5, alpha=0.85)
ax.axhline(0, color='black', linewidth=0.8)

for b, d in zip(budgets_arr, deltas):
    ax.text(b, d + (0.15 if d >= 0 else -0.35), '{:+.2f}'.format(d),
            ha='center', fontsize=9)

ax.set_xlabel('Cumulative labeled budget')
ax.set_ylabel('Accuracy difference (%)')
ax.set_title('Cosine minus Euclidean typicality')
ax.set_xticks(budgets_arr)
ax.grid(True, axis='y', linestyle='--', alpha=0.4)

plt.tight_layout()
plt.savefig(str(PLOTS_DIR / 'modified_delta_bar.pdf'), bbox_inches='tight')
plt.show()
print('Saved -> plots/modified_delta_bar.pdf')

---
## Section 8: Statistical Analysis

Two comparisons at each budget level:
1. **Original vs. Modified** — is cosine typicality significantly different?
2. **Modified vs. Random** — does the modification still beat random?

In [ ]:
def cohens_d(a, b):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    diff = a - b
    sd = diff.std(ddof=1)
    if sd == 0:
        return 0.0
    return float(diff.mean() / sd)

def interpret_d(d):
    ad = abs(d)
    if ad < 0.2: return 'negligible'
    if ad < 0.5: return 'small'
    if ad < 0.8: return 'medium'
    return 'large'

In [ ]:
if N_SEEDS < 2:
    print('WARNING: N_SEEDS < 2 -- cannot run statistical tests.')
    stat_orig_vs_cos = []
    stat_cos_vs_rnd  = []
else:
    # test 1: Original vs. Cosine (the core comparison)
    print('Paired t-test: Original (Euclidean) vs. Modified (Cosine)\n')
    header = '{:>8s}  {:>9s}  {:>9s}  {:>8s}  {:>9s}  {:>7s}  {:>9s}  {:>10s}'.format(
        'Budget', 'Orig', 'Cosine', 't-stat', 'p-value', 'Sig', "Cohen d", 'Effect')
    print(header)
    print('-' * len(header))

    stat_orig_vs_cos = []
    for i, b in enumerate(budgets_arr):
        a = agg_orig['matrix'][:, i]
        c = agg_cos['matrix'][:, i]
        t_stat, p_val = stats.ttest_rel(c, a)   # positive t => cosine > original
        d = cohens_d(c, a)
        sig = '*' if p_val < 0.05 else ('~' if p_val < 0.10 else '')
        print('{:>8d}  {:>8.2f}%  {:>8.2f}%  {:>8.3f}  {:>9.4f}  {:>7s}  {:>9.3f}  {:>10s}'.format(
            b, a.mean(), c.mean(), t_stat, p_val, sig, d, interpret_d(d)))
        stat_orig_vs_cos.append({
            'budget': int(b), 't_stat': float(t_stat),
            'p_value': float(p_val), 'cohens_d': float(d),
        })

    print('\n  * p < 0.05   ~ p < 0.10')

    # test 2: Cosine vs. Random
    print('\n\nPaired t-test: Modified (Cosine) vs. Random\n')
    print(header)
    print('-' * len(header))

    stat_cos_vs_rnd = []
    for i, b in enumerate(budgets_arr):
        c = agg_cos['matrix'][:, i]
        r = agg_rnd['matrix'][:, i]
        t_stat, p_val = stats.ttest_rel(c, r)
        d = cohens_d(c, r)
        sig = '*' if p_val < 0.05 else ('~' if p_val < 0.10 else '')
        print('{:>8d}  {:>8.2f}%  {:>8.2f}%  {:>8.3f}  {:>9.4f}  {:>7s}  {:>9.3f}  {:>10s}'.format(
            b, c.mean(), r.mean(), t_stat, p_val, sig, d, interpret_d(d)))
        stat_cos_vs_rnd.append({
            'budget': int(b), 't_stat': float(t_stat),
            'p_value': float(p_val), 'cohens_d': float(d),
        })

    print('\n  * p < 0.05   ~ p < 0.10')

    save_results(
        {'original_vs_cosine': stat_orig_vs_cos, 'cosine_vs_random': stat_cos_vs_rnd},
        str(RESULTS_DIR / 'modified_statistical_analysis.json'),
    )

In [ ]:
# statistical analysis plot
if N_SEEDS >= 2 and len(stat_orig_vs_cos) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))

    # p-values: Orig vs Cosine
    ax = axes[0]
    p_oc = [r['p_value'] for r in stat_orig_vs_cos]
    p_cr = [r['p_value'] for r in stat_cos_vs_rnd]
    ax.plot(budgets_arr, p_oc, marker='o', color=COS_COLOR, linewidth=2,
            label='Orig vs Cosine')
    ax.plot(budgets_arr, p_cr, marker='s', color=RND_COLOR, linewidth=2,
            label='Cosine vs Random')
    ax.axhline(0.05, color='red', linestyle='--', linewidth=1, label='alpha=0.05')
    ax.set_xlabel('Budget')
    ax.set_ylabel('p-value')
    ax.set_title('Paired t-test p-values')
    ax.set_xticks(budgets_arr)
    ax.legend(fontsize=8)
    ax.grid(True, linestyle='--', alpha=0.4)

    # Cohen's d: Orig vs Cosine
    ax = axes[1]
    d_oc = [r['cohens_d'] for r in stat_orig_vs_cos]
    d_cr = [r['cohens_d'] for r in stat_cos_vs_rnd]
    w = 1.5
    ax.bar(budgets_arr - w, d_oc, width=2*w, color=COS_COLOR, alpha=0.8,
           label='Cosine - Original')
    ax.bar(budgets_arr + w, d_cr, width=2*w, color=RND_COLOR, alpha=0.8,
           label='Cosine - Random')
    for thresh, lbl, ls in [(0.2, 'small', '--'), (0.5, 'medium', '-'), (0.8, 'large', ':')]:
        ax.axhline(thresh, color='gray', linestyle=ls, linewidth=1, label=lbl)
    ax.set_xlabel('Budget')
    ax.set_ylabel("Cohen's d")
    ax.set_title('Effect sizes')
    ax.set_xticks(budgets_arr)
    ax.legend(fontsize=7, ncol=2)
    ax.grid(True, axis='y', linestyle='--', alpha=0.4)

    plt.tight_layout()
    plt.savefig(str(PLOTS_DIR / 'modified_statistical.pdf'), bbox_inches='tight')
    plt.show()
    print('Saved -> plots/modified_statistical.pdf')
else:
    print('Skipping plots (need N_SEEDS >= 2).')

---
## Section 9: Summary

In [ ]:
W = 82
sep  = '-' * W
dsep = '=' * W

print(dsep)
print('{:^{}s}'.format('MODIFIED TypiClust: Euclidean vs. Cosine Typicality', W))
print('{:^{}s}'.format('CIFAR-10  |  B={}, {} rounds, {} seeds'.format(BUDGET_B, N_ROUNDS, N_SEEDS), W))
print(dsep)
print('{:>8s}  |  {:^22s}  |  {:^22s}  |  {:^14s}'.format(
    'Budget', 'Original (Euclid)', 'Modified (Cosine)', 'Random'))
print('{:>8s}  |  {:^22s}  |  {:^22s}  |  {:^14s}'.format(
    '', 'mean +/- std', 'mean +/- std', 'mean +/- std'))
print(sep)

for i, b in enumerate(budgets_arr):
    orig_str = '{:.2f} +/- {:.2f}'.format(agg_orig['mean'][i], agg_orig['std'][i])
    cos_str  = '{:.2f} +/- {:.2f}'.format(agg_cos['mean'][i],  agg_cos['std'][i])
    rnd_str  = '{:.2f} +/- {:.2f}'.format(agg_rnd['mean'][i],  agg_rnd['std'][i])
    print('{:>8d}  |  {:^22s}  |  {:^22s}  |  {:^14s}'.format(
        int(b), orig_str, cos_str, rnd_str))

print(dsep)
print('\nFinal budget ({} labels):'.format(BUDGETS[-1]))
print('  Original (Euclidean) : {:.2f}% +/- {:.2f}%'.format(
    agg_orig['mean'][-1], agg_orig['std'][-1]))
print('  Modified (Cosine)    : {:.2f}% +/- {:.2f}%'.format(
    agg_cos['mean'][-1], agg_cos['std'][-1]))
print('  Random               : {:.2f}% +/- {:.2f}%'.format(
    agg_rnd['mean'][-1], agg_rnd['std'][-1]))
print('  Delta (Cosine - Orig): {:+.2f}%'.format(
    agg_cos['mean'][-1] - agg_orig['mean'][-1]))
print('  Delta (Cosine - Rand): {:+.2f}%'.format(
    agg_cos['mean'][-1] - agg_rnd['mean'][-1]))
print(dsep)

---
## Section 10: Discussion

Fill in after running the experiments:

### Does cosine typicality improve over Euclidean?

*Your analysis here.* Consider:
- At which budget levels does the modification help/hurt?
- Are the differences statistically significant?
- Does the effect size suggest a meaningful practical difference?

### Why might the results differ (or not)?

For L2-normalised vectors on the unit sphere:
- `||a - b||_2^2 = 2 - 2 * cos(a, b)`
- So Euclidean distance and cosine distance identify the **same** nearest neighbours
- The key difference is in the aggregation:
  - **Euclidean typicality**: `1 / mean(distances)` -- the inverse makes it non-linear, amplifying the effect of one large distance among otherwise small ones
  - **Cosine typicality**: `mean(similarities)` -- linear average, so each neighbour contributes proportionally
- If all K neighbours are at similar distances, both methods agree
- If one neighbour is much farther, Euclidean typicality drops more sharply

### Practical implications

- Is the modification worth the added complexity for practitioners?
- Under what conditions might the difference become larger?
  (e.g., noisier features, higher K, different datasets)

In [ ]:
# list all outputs from this notebook
print('Generated files:')
for pattern in ['../results/modified_*.json', '../plots/modified_*.pdf']:
    for f in sorted(glob.glob(pattern)):
        size_kb = Path(f).stat().st_size / 1024
        print('  {}  ({:.1f} KB)'.format(f, size_kb))